## Create Csv file with Elo Ratings for Coaches and Teams
Season, DayNum, TeamID, (CoachElo, TeamElo) (until DayNum.Season)
Save to data/elo.csv

# Teoria rankingu Elo

System Elo opiera się na modelu probabilistycznym, który zakłada, że wyniki rywalizacji można modelować za pomocą funkcji logistycznej.

Prawdopodobieństwo, że drużyna A (z rankingiem Elo $R_A$) pokona drużynę B (z rankingiem Elo $R_B$) jest dane wzorem:

$$
P(A) = \frac{1}{1 + 10^{\frac{R_B - R_A}{400}}}
$$

Analogicznie, prawdopodobieństwo, że drużyna B wygra, to:

$$
P(B) = 1 - P(A)
$$

Przykład: Jeśli drużyna ma ranking 2000, a przeciwnik 1800, to:

$$
\begin{aligned}
P(2000 \text{ wygrywa}) &= \frac{1}{1 + 10^{\frac{1800 - 2000}{400}}} \\
&= \frac{1}{1 + 10^{\frac{-200}{400}}} \\
&= \frac{1}{1 + 10^{-0.5}} \\
&\approx \frac{1}{1 + 0.316} \\
&\approx 0.76
\end{aligned}
$$

Oznacza to, że drużyna z ELO 2000 ma około 76% szans na zwycięstwo z drużyną o ELO 1800.

In [1]:
import pandas as pd

In [2]:
def calculate_elo(df, k=32, initial_elo=1500):
    # Słownik przechowujący aktualne wartości ELO dla drużyn
    elo_ratings = {}

    # Lista do przechowywania wynikowej ramki danych
    elo_history = []

    for (season, day), games in df.groupby(['Season', 'DayNum']):
        for _, game in games.iterrows():
            w_team, l_team = game['WTeamID'], game['LTeamID']

            # Pobranie aktualnych wartości ELO (lub ustawienie domyślnej)
            w_elo = elo_ratings.get(w_team, initial_elo)
            l_elo = elo_ratings.get(l_team, initial_elo)

            # Obliczanie prawdopodobieństw wygranej
            p_w = 1 / (1 + 10 ** ((l_elo - w_elo) / 400))
            p_l = 1 - p_w

            # Aktualizacja wartości ELO
            w_elo_new = w_elo + k * (1 - p_w)
            l_elo_new = l_elo + k * (0 - p_l)

            elo_ratings[w_team] = w_elo_new
            elo_ratings[l_team] = l_elo_new

            # Zapisywanie wyników
            elo_history.append([season, day, w_team, w_elo_new])
            elo_history.append([season, day, l_team, l_elo_new])

    # Tworzenie ramki wynikowej
    elo_df = pd.DataFrame(elo_history, columns=['Season', 'DayNum', 'TeamID', 'TeamELO'])
    return elo_df

# Przykładowe dane
data = {
    'Season': [2021, 2021, 2021, 2021],
    'DayNum': [10, 20, 30, 40],
    'WTeamID': [1, 2, 1, 3],
    'LTeamID': [2, 3, 3, 1]
}
df = pd.DataFrame(data)

# Obliczenie rankingu ELO
elo_df = calculate_elo(df)
print(elo_df)

   Season  DayNum  TeamID      TeamELO
0    2021      10       1  1510.000000
1    2021      10       2  1490.000000
2    2021      20       2  1500.287744
3    2021      20       3  1489.712256
4    2021      30       1  1519.416735
5    2021      30       3  1480.295522
6    2021      40       3  1491.416786
7    2021      40       1  1508.295470


In [7]:
# Wczytanie danych
folder = '../../data/'
MenSeason = pd.read_csv(folder+'MRegularSeasonCompactResults.csv')
MenTournament = pd.read_csv(folder+'MNCAATourneyCompactResults.csv')
WomenSeason = pd.read_csv(folder+'WRegularSeasonCompactResults.csv')
WomenTournament = pd.read_csv(folder+'WNCAATourneyCompactResults.csv')
#create together df
df = pd.concat([MenSeason, MenTournament, WomenSeason, WomenTournament], ignore_index=True)
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0
...,...,...,...,...,...,...,...,...
333288,2024,147,3163,80,3425,73,A,0
333289,2024,147,3234,94,3261,87,H,0
333290,2024,151,3234,71,3163,69,N,0
333291,2024,151,3376,78,3301,59,N,0


In [8]:
#sort df by Season and DayNum
df = df.sort_values(by=['Season', 'DayNum'])
df

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,1985,20,1228,81,1328,64,N,0
1,1985,25,1106,77,1354,70,H,0
2,1985,25,1112,63,1223,56,H,0
3,1985,25,1165,70,1432,54,H,0
4,1985,25,1192,86,1447,74,H,0
...,...,...,...,...,...,...,...,...
331638,2025,120,3396,60,3150,54,A,0
331639,2025,120,3409,58,3455,46,A,0
331640,2025,120,3412,66,3408,64,H,0
331641,2025,120,3427,67,3187,48,A,0


In [10]:
df = df [['Season', 'DayNum', 'WTeamID', 'LTeamID']]
elo_df = calculate_elo(df)
elo_df

,Season,DayNum,TeamID,TeamELO
0,1985,20,1228,1510.000000
1,1985,20,1328,1490.000000
2,1985,25,1106,1510.000000
3,1985,25,1354,1490.000000
4,1985,25,1112,1510.000000
...,...,...,...,...
666581,2025,120,3408,1562.798659
666582,2025,120,3427,1685.232612
666583,2025,120,3187,1603.690091
666584,2025,120,3460,1366.578588


In [11]:
coaches = pd.read_csv(folder+'MTeamCoaches.csv')
coaches

,Season,TeamID,FirstDayNum,LastDayNum,CoachName
0,1985,1102,0,154,reggie_minton
1,1985,1103,0,154,bob_huggins
2,1985,1104,0,154,wimp_sanderson
3,1985,1106,0,154,james_oliver
4,1985,1108,0,154,davey_whitney
...,...,...,...,...,...
13528,2025,1476,0,120,chris_kraus
13529,2025,1477,0,120,jaret_von_rosenberg
13530,2025,1478,0,120,nate_champion
13531,2025,1479,0,120,gary_manchel
